# scGPT BCL6 in silico KO on post-cardiomyocytes

This notebook is a stripped-down adaptation of the official scGPT zero-shot
reference-mapping / perturbation tutorials:
https://github.com/bowang-lab/scGPT/blob/main/tutorials

Approach used here (zero-shot, no fine-tuning):

  1. Build an AnnData from the project's exported expression matrix.
  2. Run scGPT's `embed_data` / `forward` on each cell once to get the
     model's predicted expression under the original input.
  3. Re-run with BCL6's input expression silenced (knockdown_strength scaled
     down to 0 in the lowest expression bin). The decoder's predicted
     expression for all genes is the perturbed expression.
  4. Save both predicted matrices into a single `.h5ad`. This is the native
     artefact `scripts/08_run_scgpt_bcl6.py` consumes.

If you have an scGPT perturbation pipeline of your own (e.g. a fine-tuned
GEARS-style model), all you need to produce is one `.h5ad` whose `.X` (or
`.layers['perturbed']`) holds the perturbed cells x genes matrix and whose
`.var_names` and `.obs_names` match this project's gene symbols and cell IDs.
Skip this notebook and just run `scripts/08_run_scgpt_bcl6.py`.

Environment: `envs/scgpt.yml`.

In [ ]:
import os
from pathlib import Path

import numpy as np
import pandas as pd
import scanpy as sc
import anndata as ad
import yaml
import torch

In [ ]:
PROJECT_ROOT = Path(os.getcwd())
if PROJECT_ROOT.name == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent
os.chdir(PROJECT_ROOT)

with open('configs/config.yaml') as f:
    config = yaml.safe_load(f)

EXPR_PATH = Path(config['paths']['exported_expression_csv'])
META_PATH = Path(config['paths']['exported_meta_csv'])
NATIVE_OUT = Path(config['model_runners']['scgpt']['native_output_path'])
TARGET_GENE_CANDIDATES = config['perturbation']['target_gene_candidates']
KD_STRENGTH = float(config['perturbation']['knockdown_strength'])

# scGPT pretrained checkpoint paths — download from
# https://github.com/bowang-lab/scGPT#pretrained-models
# and unzip into a directory containing best_model.pt, vocab.json, args.json.
SCGPT_CKPT_DIR = Path(os.environ.get('SCGPT_CKPT_DIR', 'data/raw/scgpt_human'))

NATIVE_OUT.parent.mkdir(parents=True, exist_ok=True)
print('SCGPT_CKPT_DIR =', SCGPT_CKPT_DIR)
print('Native output target:', NATIVE_OUT)

## 1. Build AnnData from the exported matrices

In [ ]:
expr = pd.read_csv(EXPR_PATH, index_col=0)
expr.index = expr.index.astype(str)
expr.columns = expr.columns.astype(str)
meta = pd.read_csv(META_PATH, index_col=0)
if 'cell_id' in meta.columns:
    meta.index = meta['cell_id'].astype(str)
common = meta.index.intersection(expr.columns)
expr = expr.loc[:, common]
meta = meta.loc[common]

adata = ad.AnnData(
    X=expr.values.T.astype(np.float32),
    obs=meta.copy(),
    var=pd.DataFrame(index=expr.index.astype(str)),
)
adata.var['gene_name'] = adata.var_names
print(adata)

## 2. Load scGPT pretrained model

Note: scGPT's API has been moving. The block below targets the API used in the
official `Tutorial_ZeroShot.ipynb` (TransformerModel + GeneVocab). If your
installed version's API differs, swap in the matching loader from
`scgpt.tasks.cell_emb.embed_data` or `scgpt.model.TransformerModel.from_pretrained`.

In [ ]:
import json
from scgpt.model import TransformerModel
from scgpt.tokenizer import GeneVocab

vocab_file = SCGPT_CKPT_DIR / 'vocab.json'
args_file = SCGPT_CKPT_DIR / 'args.json'
model_file = SCGPT_CKPT_DIR / 'best_model.pt'
for p in [vocab_file, args_file, model_file]:
    if not p.exists():
        raise FileNotFoundError(
            f'Missing scGPT artefact: {p}. Set SCGPT_CKPT_DIR to a directory '
            'containing best_model.pt, vocab.json, and args.json.'
        )

vocab = GeneVocab.from_file(vocab_file)
with open(args_file) as f:
    model_args = json.load(f)
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Device:', device)
print('Loaded vocab size:', len(vocab))

In [ ]:
# Map gene symbols to vocab tokens; drop genes scGPT does not know about.
gene_in_vocab = np.array([g in vocab for g in adata.var_names])
print(f'{gene_in_vocab.sum()} / {len(gene_in_vocab)} genes present in scGPT vocab')
adata = adata[:, gene_in_vocab].copy()
adata.var['id_in_vocab'] = [vocab[g] for g in adata.var_names]
vocab.set_default_index(vocab['<pad>'])
print('After vocab filtering:', adata)

## 3. Zero-shot KO via input ablation

Strategy: encode each cell with original expression and then with BCL6's
binned expression set to 0 (lowest token), feed through the model and ask
the masked-value-prediction head for predicted expression of all genes.
Difference between the two reconstructions is our perturbation effect.

In [ ]:
from scgpt.preprocess import Preprocessor
from scgpt.utils import set_seed

set_seed(0)
preprocessor = Preprocessor(
    use_key='X',
    filter_gene_by_counts=False,
    filter_cell_by_counts=False,
    normalize_total=1e4,
    result_normed_key='X_normed',
    log1p=True,
    result_log1p_key='X_log1p',
    subset_hvg=False,
    binning=51,
    result_binned_key='X_binned',
)
preprocessor(adata, batch_key=None)
print('Preprocessing done. layers:', list(adata.layers.keys()))

In [ ]:
model = TransformerModel(
    ntoken=len(vocab),
    d_model=model_args['embsize'],
    nhead=model_args['nheads'],
    d_hid=model_args['d_hid'],
    nlayers=model_args['nlayers'],
    nlayers_cls=model_args.get('n_layers_cls', 3),
    n_cls=1,
    vocab=vocab,
    dropout=model_args.get('dropout', 0.2),
    pad_token='<pad>',
    pad_value=model_args.get('pad_value', -2),
    do_mvc=True,
    do_dab=False,
    use_batch_labels=False,
    domain_spec_batchnorm=False,
    explicit_zero_prob=False,
    use_fast_transformer=False,
    pre_norm=False,
)
model.load_state_dict(torch.load(model_file, map_location=device), strict=False)
model.to(device).eval()
print('Model loaded.')

In [ ]:
def predict_expression(adata, mask_gene=None, batch_size=64):
    """Run the scGPT MVC head once per cell and return cells x genes predictions.

    If `mask_gene` is given, that gene's binned input value is replaced with 0
    before the forward pass to simulate a knockdown.
    """
    binned = adata.layers['X_binned'].astype(np.int64)
    n_cells, n_genes = binned.shape
    gene_ids = np.array(adata.var['id_in_vocab'].values, dtype=np.int64)
    out = np.zeros((n_cells, n_genes), dtype=np.float32)

    if mask_gene is not None:
        mask_idx = adata.var_names.get_loc(mask_gene)
    with torch.no_grad():
        for start in range(0, n_cells, batch_size):
            end = min(start + batch_size, n_cells)
            batch = binned[start:end].copy()
            if mask_gene is not None:
                # Zero (lowest-bin) the perturbed gene so the encoder sees a knockdown.
                batch[:, mask_idx] = max(0, int(np.round((1.0 - KD_STRENGTH) * batch[:, mask_idx].mean())))
            x = torch.from_numpy(batch).to(device)
            ids = torch.from_numpy(np.tile(gene_ids, (end - start, 1))).to(device)
            src_key_padding_mask = torch.zeros_like(ids, dtype=torch.bool)
            output = model(
                src=ids,
                values=x.float(),
                src_key_padding_mask=src_key_padding_mask,
                MVC=True,
                CLS=False,
                CCE=False,
                ECS=False,
            )
            mvc_pred = output['mvc_output'].detach().cpu().numpy()
            out[start:end] = mvc_pred
    return out

In [ ]:
target_gene = next((g for g in TARGET_GENE_CANDIDATES if g in adata.var_names), None)
if target_gene is None:
    raise ValueError(f'None of {TARGET_GENE_CANDIDATES} found after vocab filtering.')
print('Target gene resolved to:', target_gene)

pred_original = predict_expression(adata, mask_gene=None)
pred_perturbed = predict_expression(adata, mask_gene=target_gene)
print('Predictions:', pred_original.shape, pred_perturbed.shape)

## 4. Save the native artefact for the project converter

In [ ]:
ad_out = ad.AnnData(
    X=pred_original.astype(np.float32),
    obs=adata.obs.copy(),
    var=adata.var.copy(),
)
ad_out.layers['perturbed'] = pred_perturbed.astype(np.float32)
ad_out.layers['original_predicted'] = pred_original.astype(np.float32)
ad_out.uns['perturbation'] = {
    'target_gene': target_gene,
    'knockdown_strength': KD_STRENGTH,
    'method': 'scgpt_zero_shot_mvc_input_mask',
}
ad_out.write_h5ad(NATIVE_OUT)
print('Wrote native scGPT artefact to', NATIVE_OUT)
print('Now run: python scripts/08_run_scgpt_bcl6.py --config configs/config.yaml')